[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/13-fine-tuning/notebooks/04-qlora-colab-only.ipynb)

# 04 - QLoRA on a small instruct model (Colab T4 GPU only)

**Before you start: Runtime → Change runtime type → T4 GPU.** This notebook needs an NVIDIA GPU because 4-bit layers come from `bitsandbytes`. It is *not* run by the course's automatic checks.

**Goal:** fine-tune `Qwen/Qwen2.5-0.5B-Instruct` with **QLoRA** (frozen base weights in 4-bit NF4 + trainable LoRA adapters) using `transformers`, `peft`, `trl` and `bitsandbytes`, on the Chai Corner dataset. Then evaluate before/after, save, merge, and optionally push to the Hub.

Expected time on a T4: about 5-10 minutes including downloads.

Versions: this notebook was written and API-checked against **transformers 5.23.0, peft 0.22.0, trl 1.18.2** (late 2026) and pins them. If you change versions, check argument names in the official docs.

## Step 1 - Install the libraries

`bitsandbytes` provides the 4-bit layers. We pin the versions the notebook was checked with. After installing, if Colab asks you to restart the session, do so and continue from Step 2.

In [ ]:
!pip install -q "transformers==5.23.0" "peft==0.22.0" "trl==1.18.2" bitsandbytes accelerate datasets

## Step 2 - Check the GPU

`torch.cuda.is_available()` must be `True`. The T4 does not support fast bfloat16, so we use **float16** for computation.

In [ ]:
import os, json, pathlib, urllib.request, time
import torch
assert torch.cuda.is_available(), "No GPU found: Runtime -> Change runtime type -> T4 GPU, then rerun."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
import transformers, peft, trl
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)

## Step 3 - Get the dataset

The same 48 training and 12 test examples as notebook 03, downloaded from the course repo.

In [ ]:
RAW = "https://raw.githubusercontent.com/vvvvvivekkk/AI-Engineer/main/13-fine-tuning/data"
for name in ["chai_corner_train.jsonl", "chai_corner_test.jsonl"]:
    if not pathlib.Path(name).exists():
        urllib.request.urlretrieve(f"{RAW}/{name}", name)

from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "chai_corner_train.jsonl", "test": "chai_corner_test.jsonl"})
print(ds)

## Step 4 - Convert to prompt / completion format

TRL understands several dataset formats. In the **conversational prompt-completion** format, `prompt` is the list of messages before the answer and `completion` is the assistant message. With this format `SFTTrainer` computes the loss **only on the completion** (`completion_only_loss` defaults to on), which is the loss masking we did by hand in notebook 03.

In [ ]:
def to_prompt_completion(ex):
    return {"prompt": ex["messages"][:-1], "completion": ex["messages"][-1:]}

ds = ds.map(to_prompt_completion, remove_columns=["messages"])
print(ds["train"][0])

## Step 5 - Load the model in 4-bit

`BitsAndBytesConfig`:
- `load_in_4bit=True` - store the frozen weights in 4 bits;
- `bnb_4bit_quant_type="nf4"` - the NormalFloat levels from notebook 02;
- `bnb_4bit_use_double_quant=True` - also compress the scale numbers;
- `bnb_4bit_compute_dtype=torch.float16` - dequantise to float16 for the maths (T4-friendly).

`device_map="auto"` places the model on the GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # try Qwen/Qwen2.5-1.5B-Instruct for a bigger model
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                             device_map="auto", dtype=torch.float16)
print(f"GPU memory used by the 4-bit model: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 6 - Measure BEFORE training

The same style score as notebook 03: does the answer start with "Chai Corner here!" and contain "Have a warm day!"? We use all 12 test questions and greedy decoding.

In [ ]:
PREFIX, SUFFIX = "Chai Corner here!", "Have a warm day!"
test_rows = list(ds["test"])

@torch.no_grad()
def reply(m, prompt_msgs, max_new_tokens=48):
    enc = tok.apply_chat_template(prompt_msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(m.device)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def style_score(m):
    m.eval()
    answers = [reply(m, r["prompt"]) for r in test_rows]
    return sum(a.startswith(PREFIX) and SUFFIX in a for a in answers) / len(answers), answers

before_style, before_answers = style_score(model)
print(f"BEFORE style score: {before_style:.0%}")
print("Example:", before_answers[0])

## Step 7 - Prepare for k-bit training and define LoRA

`prepare_model_for_kbit_training` (from `peft`) gets a quantised model ready: it freezes the base, keeps small layers such as layer norms in higher precision, and turns on **gradient checkpointing** (saves memory by recomputing activations).

`target_modules="all-linear"` puts adapters on every linear layer (attention and MLP), a common QLoRA choice.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                         target_modules="all-linear", task_type="CAUSAL_LM")
print(peft_config)

## Step 8 - Configure and create the SFTTrainer

Key `SFTConfig` settings:
- `num_train_epochs=3`, `per_device_train_batch_size=4`, `gradient_accumulation_steps=2` → effective batch 8;
- `learning_rate=2e-4` - a typical LoRA learning rate;
- `max_length=256` - longer examples would be truncated (ours are short);
- `fp16=True` - mixed precision on the T4 (do **not** use `bf16` on a T4);
- `optim="paged_adamw_8bit"` - the memory-saving paged optimizer from QLoRA;
- `report_to="none"` - do not send logs to external services.

We pass `peft_config`, so the trainer wraps the model with LoRA for us.

In [ ]:
from trl import SFTConfig, SFTTrainer
args = SFTConfig(
    output_dir="qlora-chai",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    logging_steps=2,
    max_length=256,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tok, peft_config=peft_config)
trainer.model.print_trainable_parameters()

## Step 9 - Train

Watch the training loss fall and the evaluation loss after each epoch. If the loss shows `nan`, restart and set `fp16=False` (slower but safer).

In [ ]:
t0 = time.time()
trainer.train()
print(f"training took {time.time() - t0:.0f}s; peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## Step 10 - Measure AFTER training

Same 12 questions. Also read the answers against the references - check the facts, not only the style.

In [ ]:
after_style, after_answers = style_score(trainer.model)
print(f"style score: {before_style:.0%} -> {after_style:.0%}")
print("eval loss:", trainer.evaluate()["eval_loss"])
for r, a in list(zip(test_rows, after_answers))[:5]:
    print("\nQ:", r["prompt"][-1]["content"]); print("A:", a); print("ref:", r["completion"][0]["content"])

## Step 11 - Save the adapter

Only the LoRA weights are saved (a few tens of MB at most), plus the tokenizer.

In [ ]:
ADAPTER_DIR = "qlora-chai-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}

## Step 12 - Merge into a 16-bit model

You cannot cleanly merge into 4-bit weights. Instead, load a fresh **float16** copy of the base model, attach the adapter, and merge. We free the training model first to make room on the GPU.

In [ ]:
import gc
from peft import PeftModel
del trainer, model
gc.collect(); torch.cuda.empty_cache()
base16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base16, ADAPTER_DIR).merge_and_unload()
print("merged answer:", reply(merged, test_rows[0]["prompt"]))
merged.save_pretrained("qlora-chai-merged")
tok.save_pretrained("qlora-chai-merged")

## Step 13 - Push to the Hugging Face Hub (optional, guarded)

Add your write token as a Colab **secret** named `HF_TOKEN` (key icon in the left sidebar) or as an environment variable. Never paste it into a cell. Set `PUSH = True` and change `REPO_ID` to your username. Pushing the adapter is usually enough; others load it on top of the same base model. Remember: the base model's licence applies to your fine-tune too.

In [ ]:
PUSH = False
REPO_ID = "your-username/chai-corner-qwen2.5-0.5b-qlora"
token = os.getenv("HF_TOKEN")
if token is None:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if PUSH and token:
    from huggingface_hub import HfApi
    HfApi(token=token).create_repo(REPO_ID, private=True, exist_ok=True)
    HfApi(token=token).upload_folder(folder_path=ADAPTER_DIR, repo_id=REPO_ID)
    print("uploaded adapter to https://huggingface.co/" + REPO_ID)
else:
    print("Skipping upload (needs HF_TOKEN and PUSH = True).")

## What you learned
- QLoRA = load the base in 4-bit NF4 + train LoRA adapters; a T4 is enough for small models.
- TRL's `SFTTrainer` handles the chat template and completion-only loss for prompt/completion data.
- Evaluate before/after on held-out data and read the outputs.
- Merge into a 16-bit base, not the 4-bit one; share adapters with a token from a secret.